In [ ]:
import numpy as np
import os
import sys

sys.path.append('../../..')
path=os.path.abspath(os.path.join(os.getcwd(), "../../../outputs/linear"))

import torch
import torchsde
from torch import nn, profiler, Tensor, nan, vmap
from tqdm import tqdm


import gc
from core.utils import get_gpu_memory, normal, mass_append,  generate_system_canon, load_clean_state_dict, sample_linear, compute_gradnorm,save_model, loss_compute
from core.policies_lqr import   HyperLinearControl,  find_params
from configs.linear_cfg import eval_cfg
from IPython.display import clear_output

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")




In [ ]:

# --- 1. Evaluation Function ---
def evaluate_lqr_strategy(layers, config, checkpoint_path):
    """
    Standardized evaluation for Stochastic LQR Control.
    Uses Welford's algorithm for numerically stable running statistics.
    """
    # Initialize Model Environment
    A, B = generate_system_canon(config.dim)
    At = torch.as_tensor(A, device=device, dtype=torch.float32)
    Bt = torch.as_tensor(B, device=device, dtype=torch.float32)
    
    model = HyperLinearControl(
        At, Bt, config.batch_size, device, layers, 
        config.strategy, config.skip, config.activation, 
        config.clip, config.std_scales
    ).to(device)
    
    # Load Weights
    if os.path.exists(checkpoint_path):
        load_clean_state_dict(model, checkpoint_path)
    else:
        print(f"Warning: Checkpoint not found at {checkpoint_path}")
        
    model.eval()
    
    # Setup Simulation Parameters
    ts_exp = torch.linspace(0, config.t_total, config.t_size).to(device)
    dt = ts_exp[1] - ts_exp[0]
    coremat = At.unsqueeze(0).repeat(config.batch_size, 1, 1)
    
    # Welford Tracking Variables
    count = 0
    mean = 0.0
    M2 = 0.0

    # --- 2. Inference Block ---
    # Using inference_mode context manager for better performance than no_grad
    with torch.inference_mode():
        pbar = tqdm(range(config.test_iters), desc=f"Eval L{layers}", leave=False)
        
        for j in pbar:
            # Sample initial conditions
            amat, x0 = sample_linear(config.dim, config.batch_size, device)   
            ini_enr = torch.zeros(config.batch_size, 1, device=device)
            ys_init = torch.cat((x0, ini_enr), dim=1)
            
            # Update dynamic system matrices
            coremat[:, -1, :] = amat
            model.At = coremat
            model.poly.amat = amat

            # Solve SDE
            ys_tray = torchsde.sdeint_adjoint(
                model, ys_init, ts_exp, method='reversible_heun', dt=dt
            )
            
            # Calculate Batch Cost
            # Cost = Terminal State Squared Norm + (Weight * Accumulated Control Energy)
            terminal_norm = torch.sum(ys_tray[-1, :, :config.dim]**2, dim=1)
            energy_cost = config.weight * ys_tray[-1, :, -1]   
            batch_costs = (terminal_norm + energy_cost).cpu().numpy()

            # Update Statistics (Welford's Batch Update)
            for x in batch_costs:
                count += 1
                delta = x - mean
                mean += delta / count
                delta2 = x - mean
                M2 += delta * delta2
            
            # Monitoring
            
            curr_ste = np.sqrt(M2 / (count - 1)) / np.sqrt(count) if count > 1 else 0
            pbar.set_postfix({"Mean": f"{mean:.4f}", "STE": f"{curr_ste:.4e}"})

            # Occasional Cleanup
            if j % 500 == 0:
                gc.collect()
                torch.cuda.empty_cache()

    # Final Metrics
    variance = M2 / (count - 1)
    ste = np.sqrt(variance) / np.sqrt(count)
    
    return mean, ste

# --- 3. Main Execution Loop ---
global_means = []
global_stds = []


for layers in range(1, 5):
    ckpt = f"{path}/{eval_cfg.strategy}ema{layers}.pth"
    m, s = evaluate_lqr_strategy(layers, eval_cfg, ckpt)
    
    global_means.append(m)
    global_stds.append(s)
    
    print(f"\nFinal Result for {layers} Layers:")
    print(f"Mean Cost: {m:.6f} ± {s:.6f} (STE)")
    
    # Final cleanup before next model load
    gc.collect()
    torch.cuda.empty_cache()